# 01 — Exploratory Data Analysis

Loads the processed train/val/test parquets and visualises:
- Dataset source distribution
- Severity / priority distribution
- Team routing distribution
- Over-escalation rate overall and by source
- Resolution time histogram (where available)
- Text length distributions
- 5 example over-escalation tickets

**Run `src/data/load_datasets.py` first to generate the parquet files.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROCESSED = '../data/processed'

In [ ]:
train = pd.read_parquet(f'{PROCESSED}/train.parquet')
val   = pd.read_parquet(f'{PROCESSED}/val.parquet')
test  = pd.read_parquet(f'{PROCESSED}/test.parquet')
df    = pd.concat([train, val, test], ignore_index=True)

print(f'Total rows : {len(df):,}')
print(f'Train      : {len(train):,}')
print(f'Val        : {len(val):,}')
print(f'Test       : {len(test):,}')
df.head(3)

## 1. Dataset source distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
counts = df['source'].value_counts()
counts.plot.barh(ax=ax, color=sns.color_palette('muted'))
ax.set_xlabel('Number of bug reports')
ax.set_title('Records per dataset source')
for i, (v, k) in enumerate(zip(counts, counts.index)):
    ax.text(v + 50, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../results/source_dist.png')
plt.show()

## 2. Priority / severity distribution

In [ ]:
priority_order = ['P0', 'P1', 'P2', 'P3', 'P4']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall
counts = df['priority'].value_counts().reindex(priority_order, fill_value=0)
colors = ['#d62728', '#ff7f0e', '#dbdb8d', '#98df8a', '#aec7e8']
counts.plot.bar(ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Priority distribution (all sources)')
axes[0].set_xlabel('Priority')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=8)

# By source (stacked)
pivot = df.groupby(['source', 'priority']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=priority_order, fill_value=0)
pivot.plot.bar(ax=axes[1], stacked=True, color=colors, edgecolor='white')
axes[1].set_title('Priority distribution by source')
axes[1].set_xlabel('Source')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Priority', bbox_to_anchor=(1.02, 1))

plt.tight_layout()
plt.savefig('../results/priority_dist.png')
plt.show()

## 3. Team routing distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
team_counts = df['team'].value_counts()
team_counts.plot.bar(ax=ax, color=sns.color_palette('Set2', len(team_counts)), edgecolor='white')
ax.set_title('Team routing label distribution')
ax.set_xlabel('Team')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
# Annotate unknown rate prominently
unknown_pct = 100 * team_counts.get('unknown', 0) / len(df)
ax.set_title(f'Team routing label distribution  (unknown: {unknown_pct:.1f}%)')
plt.tight_layout()
plt.savefig('../results/team_dist.png')
plt.show()

## 4. Over-escalation rate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Overall pie
n_pos = df['is_over_escalated'].sum()
n_neg = len(df) - n_pos
axes[0].pie([n_neg, n_pos], labels=[f'Not escalated\n{n_neg:,}', f'Over-escalated\n{n_pos:,}'],
            colors=['#aec7e8', '#d62728'], autopct='%1.1f%%', startangle=140)
axes[0].set_title('Overall class balance')

# Rate by source
src_rate = df.groupby('source')['is_over_escalated'].mean() * 100
src_rate.plot.bar(ax=axes[1], color=sns.color_palette('muted'), edgecolor='white')
axes[1].set_title('Over-escalation rate by source')
axes[1].set_ylabel('% of tickets')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

# Rate by priority (only meaningful for P0/P1 where escalation is possible)
pri_rate = df.groupby('priority')['is_over_escalated'].mean() * 100
pri_rate.reindex(['P0', 'P1', 'P2', 'P3', 'P4'], fill_value=0).plot.bar(
    ax=axes[2], color=['#d62728','#ff7f0e','#dbdb8d','#98df8a','#aec7e8'], edgecolor='white')
axes[2].set_title('Over-escalation rate by priority')
axes[2].set_ylabel('% of tickets')
axes[2].tick_params(axis='x', rotation=0)
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.savefig('../results/escalation_dist.png')
plt.show()

print(f'Total over-escalation positives : {n_pos:,} ({100*n_pos/len(df):.1f}%)')
print(f'Train split positives           : {train["is_over_escalated"].sum():,}')
print(f'Val split positives             : {val["is_over_escalated"].sum():,}')
print(f'Test split positives            : {test["is_over_escalated"].sum():,}')

## 5. Resolution time histogram (Apache source only)

In [ ]:
apache = df[df['source'] == 'apache'].copy()
apache = apache[apache['resolution_time_days'].notna()]
apache = apache[apache['resolution_time_days'] >= 0]

if len(apache) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    apache['resolution_time_days'].clip(upper=365).hist(
        bins=50, ax=ax, color='#5b9bd5', edgecolor='white')
    ax.axvline(7, color='red', linestyle='--', label='7-day threshold')
    ax.set_xlabel('Resolution time (days, capped at 365)')
    ax.set_ylabel('Count')
    ax.set_title('Apache bug resolution time distribution')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/resolution_time.png')
    plt.show()
    print(f'Median resolution time: {apache["resolution_time_days"].median():.0f} days')
else:
    print('No Apache records with resolution_time_days — skipping.')

## 6. Text length distributions

In [ ]:
df['title_words'] = df['title'].str.split().str.len()
df['body_words']  = df['body'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['title_words'].clip(upper=60).hist(bins=40, ax=axes[0], color='#5b9bd5', edgecolor='white')
axes[0].set_title('Title word count (capped at 60)')
axes[0].set_xlabel('Words')

df['body_words'].clip(upper=500).hist(bins=50, ax=axes[1], color='#ed7d31', edgecolor='white')
axes[1].set_title('Body word count (capped at 500)')
axes[1].set_xlabel('Words')
axes[1].axvline(400, color='red', linestyle='--', label='400-word truncation')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/text_lengths.png')
plt.show()

print(f'Title — mean: {df["title_words"].mean():.1f}  median: {df["title_words"].median():.0f}  max: {df["title_words"].max()}')
print(f'Body  — mean: {df["body_words"].mean():.1f}  median: {df["body_words"].median():.0f}  max: {df["body_words"].max()}')

## 7. Example over-escalation tickets

Manually review these to assess label quality.

In [ ]:
positives = df[df['is_over_escalated']].sample(5, random_state=42)

for i, (_, row) in enumerate(positives.iterrows()):
    print(f"{'='*70}")
    print(f"Example {i+1}  |  source={row['source']}  |  priority={row['priority']}  |  team={row['team']}")
    print(f"Resolution: {row.get('resolution', 'N/A')}")
    print(f"Title: {row['title']}")
    body_preview = row['body'][:300] if row['body'] else '(no body)'
    print(f"Body (first 300 chars): {body_preview}")
    print()

## 8. Split summary table

In [ ]:
summary_rows = []
for name, split in [('train', train), ('val', val), ('test', test)]:
    n_esc = split['is_over_escalated'].sum()
    summary_rows.append({
        'Split'   : name,
        'Total'   : len(split),
        'Escalation +': n_esc,
        'Esc. %'  : f'{100*n_esc/len(split):.1f}%',
        'P0'      : (split['priority'] == 'P0').sum(),
        'P1'      : (split['priority'] == 'P1').sum(),
        'P2'      : (split['priority'] == 'P2').sum(),
        'P3'      : (split['priority'] == 'P3').sum(),
        'P4'      : (split['priority'] == 'P4').sum(),
    })

pd.DataFrame(summary_rows).set_index('Split')